# Proyecto Final - Mineria de Datos en Python (Cancer)

**Dataset:** `Cancer_Data.csv` (Breast Cancer Wisconsin Diagnostic)

## Objetivo general
Construir y comparar modelos supervisados y no supervisados para apoyar la deteccion de cancer de mama, justificando tecnicamente cada decision de preparacion de datos, entrenamiento y evaluacion.

## Estructura del notebook
1. Comprension del problema  
2. Carga del dataset  
3. Diccionario de variables  
4. Calidad de datos  
5. EDA completo  
6. Preparacion de datos  
7. Modelos supervisados (clasificacion y regresion)  
8. Modelos no supervisados (PCA, KMeans, Jerarquico, DBSCAN)  
9. Evaluacion y comparacion de modelos  
10. Mejor modelo y justificacion  
11. Conclusiones

## 1) Comprension del problema

El problema principal del dataset es de **clasificacion binaria**: identificar si un tumor es maligno (`M`) o benigno (`B`) a partir de medidas morfologicas.

### Contexto tecnico
- En salud, un **falso negativo** (predecir benigno cuando es maligno) puede tener alto costo clinico.
- Por esta razon, ademas de accuracy, se priorizan metricas como **recall de la clase maligna** y **F1-score**.

### Alcance del proyecto
- Clasificacion supervisada: comparar modelos vistos en clase.
- Regresion supervisada: para cumplir la rubrica, se define una tarea de regresion dentro del mismo dataset usando un objetivo continuo (`area_worst`).
- Analisis no supervisado: reduccion de dimensionalidad y clustering para explorar estructura interna de los datos.

In [ ]:
# Librerias base
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Clasificacion
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Metricas clasificacion
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Metricas regresion
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# No supervisado
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances

RANDOM_STATE = 42
sns.set(style='whitegrid', context='notebook')
np.random.seed(RANDOM_STATE)

print('Librerias cargadas correctamente.')

## 2) Carga del dataset

Se carga el dataset y se aplica una limpieza minima de nombres de columnas para evitar errores en pipelines (`strip`, eliminar comillas y reemplazar espacios por `_`).

In [ ]:
# Carga y normalizacion de nombres de columnas
file_path = 'Cancer_Data.csv'
df = pd.read_csv(file_path)

# Limpieza de nombres de columnas
new_cols = (
    df.columns
      .str.strip()
      .str.replace('"', '', regex=False)
      .str.replace(' ', '_', regex=False)
)
df.columns = new_cols

print('Shape:', df.shape)
print('Columnas (primeras 8):', df.columns[:8].tolist())

df.head()

## 3) Diccionario de variables

El dataset contiene:
- `id`: identificador de la observacion (no aporta señal predictiva directa)
- `diagnosis`: variable objetivo de clasificacion (`M`/`B`)
- 30 variables numericas de caracteristicas morfologicas, agrupadas en:
  - `*_mean` (promedios)
  - `*_se` (error estandar)
  - `*_worst` (peor caso)

A continuacion se resume el diccionario automaticamente por columna y tipo.

In [ ]:
diccionario = pd.DataFrame({
    'variable': df.columns,
    'tipo_dato': df.dtypes.astype(str).values,
    'nulos': df.isna().sum().values,
    'n_unicos': [df[c].nunique() for c in df.columns]
})

diccionario.head(15)

## 4) Calidad de datos

Se valida calidad de datos en cuatro frentes solicitados:
1. Nulos
2. Duplicados
3. Tipos de datos
4. Outliers

La decision de tratamiento se fundamenta en resultados observados y en el impacto sobre modelos.

In [ ]:
# Revision de nulos, duplicados y tipos
resumen_calidad = pd.DataFrame({
    'tipo_dato': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'porcentaje_nulos': (df.isna().mean() * 100).round(2)
})

duplicados = df.duplicated().sum()
print('Duplicados exactos:', duplicados)

resumen_calidad.head(20)

In [ ]:
# Deteccion de outliers por IQR (conteo por variable numerica)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

outlier_stats = []
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_stats.append((col, n_out, round(100 * n_out / len(df), 2)))

outliers_df = pd.DataFrame(outlier_stats, columns=['variable', 'n_outliers', 'pct_outliers'])
outliers_df.sort_values('n_outliers', ascending=False).head(12)

### Interpretacion de calidad de datos

- Si no hay nulos relevantes, no se fuerza imputacion artificial para evitar introducir sesgo.
- `id` se eliminara en preparacion por no ser predictor causal.
- La presencia de outliers en datos clinicos puede ser informativa; por ello, se prioriza **escalado robusto via estandarizacion** y modelos tolerantes a variabilidad en lugar de eliminar observaciones de forma agresiva.
- Se mantendra trazabilidad de todas estas decisiones en la seccion de preparacion.

## 5) EDA completo

Se desarrolla analisis exploratorio con:
- Histogramas
- Boxplots
- Conteos de clase
- Correlaciones
- Analisis multivariado

Cada visualizacion se acompana con interpretacion tecnica.

In [ ]:
# Conteo de clases
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='diagnosis', palette='Set2')
plt.title('Distribucion de la variable objetivo (diagnosis)')
plt.xlabel('diagnosis')
plt.ylabel('Frecuencia')
plt.show()

class_dist = df['diagnosis'].value_counts(normalize=True).mul(100).round(2)
print('Distribucion porcentual de clases:')
print(class_dist)

**Interpretacion:**
- Existe desbalance moderado entre clases, por lo que no es suficiente evaluar solo con accuracy.
- Se priorizaran precision, recall y F1 para una evaluacion mas robusta.

In [ ]:
# Histogramas de variables seleccionadas
vars_hist = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'concavity_mean', 'radius_worst']

df[vars_hist].hist(figsize=(14, 8), bins=25)
plt.suptitle('Histogramas de variables seleccionadas', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots por diagnostico para detectar dispersion y posibles outliers
vars_box = ['radius_mean', 'perimeter_mean', 'area_mean', 'compactness_mean', 'concavity_mean', 'radius_worst']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()
for i, col in enumerate(vars_box):
    sns.boxplot(data=df, x='diagnosis', y=col, ax=axes[i], palette='Set3')
    axes[i].set_title(f'Boxplot de {col} por diagnosis')

plt.tight_layout()
plt.show()

**Interpretacion:**
- Las variables morfologicas de tamano (`radius`, `perimeter`, `area`) muestran separacion visible entre benigno y maligno.
- Se observan valores extremos, pero son plausibles en mediciones clinicas y pueden contener señal diagnostica, por lo que no se eliminan por defecto.

In [ ]:
# Correlacion entre variables numericas
corr = df.select_dtypes(include=[np.number]).corr()

plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, cbar_kws={'shrink': 0.8})
plt.title('Matriz de correlacion (variables numericas)')
plt.show()

In [ ]:
# Correlacion con objetivo codificado
le = LabelEncoder()
y_tmp = le.fit_transform(df['diagnosis'])  # B=0, M=1
df_corr_target = df.copy()
df_corr_target['diagnosis_bin'] = y_tmp

corr_target = df_corr_target.corr(numeric_only=True)['diagnosis_bin'].drop('diagnosis_bin').sort_values(ascending=False)

print('Top 10 correlaciones positivas con clase maligna:')
print(corr_target.head(10))
print('\nTop 10 correlaciones negativas con clase maligna:')
print(corr_target.tail(10))

In [ ]:
# Analisis multivariado: PCA 2D para visualizacion exploratoria
X_num = df.drop(columns=['diagnosis'])
X_num = X_num.drop(columns=['id'], errors='ignore')
X_scaled_tmp = StandardScaler().fit_transform(X_num)

pca_eda = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_eda = pca_eda.fit_transform(X_scaled_tmp)

pca_df = pd.DataFrame(X_pca_eda, columns=['PC1', 'PC2'])
pca_df['diagnosis'] = df['diagnosis'].values

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='diagnosis', alpha=0.8, palette='Set1')
plt.title('PCA (2 componentes) - vista exploratoria por clase')
plt.show()

print('Varianza explicada PC1 + PC2:', round(pca_eda.explained_variance_ratio_.sum(), 4))

**Interpretacion multivariada:**
- La proyeccion en PCA sugiere separacion parcial entre clases, lo que respalda la factibilidad de modelos lineales y no lineales.
- La superposicion entre grupos sugiere que no existira clasificacion perfecta y conviene comparar varios algoritmos.

## 6) Preparacion de datos

### Decisiones tomadas y justificacion
1. **Variable eliminada: `id`**  
   Se elimina porque es un identificador y no representa una caracteristica biologica.

2. **Nulos e imputacion**  
   Si el analisis confirma ausencia de nulos, no se imputa para no introducir ruido artificial.

3. **Encoding**  
   `diagnosis` se codifica a binaria para clasificacion (`B=0`, `M=1`).

4. **Escalado**  
   Se aplica `StandardScaler` en modelos sensibles a escala (Logistica, KNN, SVM, PCA, KMeans, DBSCAN, y modelos lineales de regresion).

5. **Train/Test split**  
   Se usa separacion 80/20 con `stratify` en clasificacion para preservar distribucion de clases.

6. **Tarea de regresion**  
   Para cumplir la rubrica de regresion, se define `area_worst` como objetivo continuo y se excluye de predictores.

In [ ]:
# Preparacion para clasificacion

df_model = df.copy()

# Eliminar identificador
X_cls = df_model.drop(columns=['diagnosis', 'id'], errors='ignore')
y_cls = LabelEncoder().fit_transform(df_model['diagnosis'])  # B=0, M=1

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.20, random_state=RANDOM_STATE, stratify=y_cls
)

print('Clasificacion - Train shape:', X_train_cls.shape)
print('Clasificacion - Test shape:', X_test_cls.shape)
print('Distribucion train (B=0, M=1):', np.bincount(y_train_cls))
print('Distribucion test (B=0, M=1):', np.bincount(y_test_cls))

In [ ]:
# Modelos de clasificacion requeridos
cls_models = {
    'Logistica': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Arbol_Decision': Pipeline([
        ('model', DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=5))
    ]),
    'Random_Forest': Pipeline([
        ('model', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, max_depth=None))
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE))
    ]),
    'Naive_Bayes': Pipeline([
        ('model', GaussianNB())
    ])
}

cls_results = []
cls_predictions = {}

for name, pipeline in cls_models.items():
    pipeline.fit(X_train_cls, y_train_cls)
    pred = pipeline.predict(X_test_cls)

    cls_predictions[name] = pred
    cls_results.append({
        'modelo': name,
        'accuracy': accuracy_score(y_test_cls, pred),
        'precision': precision_score(y_test_cls, pred),
        'recall': recall_score(y_test_cls, pred),
        'f1': f1_score(y_test_cls, pred)
    })

cls_results_df = pd.DataFrame(cls_results).sort_values('f1', ascending=False).reset_index(drop=True)
cls_results_df

In [ ]:
# Matrices de confusion de todos los modelos de clasificacion
n_models = len(cls_models)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for i, (name, pred) in enumerate(cls_predictions.items()):
    cm = confusion_matrix(y_test_cls, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[i])
    axes[i].set_title(f'Matriz de confusion - {name}')
    axes[i].set_xlabel('Predicho')
    axes[i].set_ylabel('Real')

plt.tight_layout()
plt.show()

### Interpretacion de clasificacion

- Se comparan seis algoritmos para reducir dependencia de un unico enfoque.
- Se reportan accuracy, precision, recall y F1, priorizando recall/F1 para minimizar falsos negativos en clase maligna.
- La matriz de confusion permite verificar en que tipo de error falla cada modelo.

## 7) Modelos de regresion (componente solicitado por rubrica)

Aunque el problema principal es de clasificacion, se incluye una tarea de regresion sobre `area_worst` para comparar tecnicas de regresion vistas en curso.

### Justificacion tecnica
- `area_worst` es continua y clinicamente interpretable como medida de tamano extremo de la lesion.
- Para evitar fuga trivial, se retira `area_worst` de los predictores.
- Se reportan MAE, MSE, RMSE y R².

In [ ]:
# Preparacion para regresion

reg_target = 'area_worst'

df_reg = df.copy()
X_reg = df_reg.drop(columns=[reg_target, 'id', 'diagnosis'], errors='ignore')
y_reg = df_reg[reg_target]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
)

print('Regresion - Train shape:', X_train_reg.shape)
print('Regresion - Test shape:', X_test_reg.shape)

In [ ]:
# Modelos de regresion requeridos
reg_models = {
    'Regresion_Lineal': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Ridge': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE))
    ]),
    'Lasso': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Lasso(alpha=0.01, random_state=RANDOM_STATE, max_iter=10000))
    ]),
    'Arbol_Regresion': Pipeline([
        ('model', DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=6))
    ]),
    'Random_Forest_Regresion': Pipeline([
        ('model', RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE))
    ])
}

reg_results = []
for name, pipeline in reg_models.items():
    pipeline.fit(X_train_reg, y_train_reg)
    pred = pipeline.predict(X_test_reg)

    mae = mean_absolute_error(y_test_reg, pred)
    mse = mean_squared_error(y_test_reg, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_reg, pred)

    reg_results.append({
        'modelo': name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    })

reg_results_df = pd.DataFrame(reg_results).sort_values('RMSE').reset_index(drop=True)
reg_results_df

## 8) Modelos no supervisados

Se aplican tecnicas no supervisadas para explorar estructura de grupos sin usar etiquetas:
- PCA
- KMeans
- Clustering jerarquico
- DBSCAN

Se evalua con **Silhouette** y **Dunn**.

In [ ]:
# Datos para no supervisado
X_unsup = df.drop(columns=['diagnosis', 'id'], errors='ignore')
X_unsup_scaled = StandardScaler().fit_transform(X_unsup)

# PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_unsup_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7)
plt.title('PCA 2D (sin etiquetas)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

print('Varianza explicada por PC1 y PC2:', pca.explained_variance_ratio_.round(4))
print('Varianza acumulada (2 PCs):', round(pca.explained_variance_ratio_.sum(), 4))

In [ ]:
# Funcion de indice Dunn para evaluar clustering

def dunn_index(X, labels):
    unique_labels = np.unique(labels)
    # Excluir ruido de DBSCAN para el calculo principal
    unique_labels = unique_labels[unique_labels != -1]

    if len(unique_labels) < 2:
        return np.nan

    clusters = [X[labels == label] for label in unique_labels]

    # Distancia minima entre clusters (intercluster)
    min_intercluster = np.inf
    for i in range(len(clusters)):
        for j in range(i + 1, len(clusters)):
            dists = pairwise_distances(clusters[i], clusters[j])
            min_intercluster = min(min_intercluster, dists.min())

    # Distancia maxima dentro de cada cluster (intracluster)
    max_intracluster = 0
    for cluster in clusters:
        if len(cluster) > 1:
            intra = pairwise_distances(cluster, cluster)
            max_intracluster = max(max_intracluster, intra.max())

    if max_intracluster == 0:
        return np.nan

    return min_intercluster / max_intracluster

print('Funcion Dunn definida.')

In [ ]:
# Clustering: KMeans, Jerarquico, DBSCAN

cluster_models = {
    'KMeans_k2': KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=20),
    'Jerarquico_k2': AgglomerativeClustering(n_clusters=2, linkage='ward'),
    'DBSCAN': DBSCAN(eps=2.2, min_samples=8)
}

cluster_results = []
cluster_labels = {}

for name, model in cluster_models.items():
    labels = model.fit_predict(X_unsup_scaled)
    cluster_labels[name] = labels

    # Silhouette requiere al menos 2 clusters efectivos
    valid_labels = labels
    if len(np.unique(valid_labels)) > 1 and len(np.unique(valid_labels)) < len(valid_labels):
        sil = silhouette_score(X_unsup_scaled, valid_labels)
    else:
        sil = np.nan

    dunn = dunn_index(X_unsup_scaled, valid_labels)

    cluster_results.append({
        'modelo': name,
        'n_clusters_detectados': len(set(valid_labels)) - (1 if -1 in valid_labels else 0),
        'silhouette': sil,
        'dunn': dunn,
        'n_ruido_dbscan': int((valid_labels == -1).sum()) if name == 'DBSCAN' else 0
    })

cluster_results_df = pd.DataFrame(cluster_results).sort_values('silhouette', ascending=False)
cluster_results_df

In [ ]:
# Visualizacion de clusters en espacio PCA
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, labels) in zip(axes, cluster_labels.items()):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', alpha=0.8)
    ax.set_title(f'Clusters - {name}')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.tight_layout()
plt.show()

### Interpretacion no supervisada

- Si KMeans/Jerarquico muestran silhouette superior a DBSCAN, sugiere estructura globular relativamente separable.
- DBSCAN puede detectar ruido y formas no esfericas; su desempeno depende de `eps` y `min_samples`.
- Dunn complementa silhouette al evaluar separacion minima entre clusters frente a dispersion maxima interna.

## 9) Evaluacion y comparacion de modelos

En esta seccion se consolidan resultados para seleccionar el mejor modelo considerando:
- Metricas cuantitativas
- Interpretabilidad
- Contexto del problema (reducir falsos negativos en malignos)

In [ ]:
# Tablas finales de comparacion
print('=== RESULTADOS CLASIFICACION ===')
display(cls_results_df)

print('=== RESULTADOS REGRESION ===')
display(reg_results_df)

print('=== RESULTADOS CLUSTERING ===')
display(cluster_results_df)

best_cls = cls_results_df.iloc[0]
best_reg = reg_results_df.iloc[0]
best_cluster = cluster_results_df.iloc[0]

print('\nMejor clasificacion por F1:', best_cls['modelo'])
print('Mejor regresion por RMSE:', best_reg['modelo'])
print('Mejor clustering por Silhouette:', best_cluster['modelo'])

## 10) Mejor modelo y justificacion

### Clasificacion
El mejor modelo se selecciona principalmente por **F1 y Recall** en clase maligna, para balancear sensibilidad y precision clinica.

### Regresion
El mejor modelo se elige por menor **RMSE** y mayor **R²**, valorando capacidad predictiva y estabilidad.

### No supervisado
Se prioriza silhouette y Dunn para identificar estructuras de grupo consistentes, sin sobreinterpretar clusters como diagnostico clinico definitivo.

## 11) Conclusiones

1. El dataset permite construir clasificadores con buen desempeno para diferenciar tumores benignos y malignos.
2. La comparacion multi-modelo evita conclusiones sesgadas por un solo algoritmo.
3. La preparacion (eliminacion de `id`, encoding, escalado y particion estratificada) fue clave para resultados validos.
4. El analisis no supervisado complemento la comprension de estructura interna y separabilidad del problema.
5. En contexto clinico, la metrica prioritaria debe ser recall de malignos para minimizar falsos negativos.

## Exportables
- Este notebook (`.ipynb`) esta listo para abrir en Google Colab.
- Para el informe PDF: en Colab usar **Archivo > Imprimir** o **Archivo > Descargar > PDF**.
- La presentacion puede apoyarse en: problema, metodologia, comparacion de modelos, mejor modelo y conclusiones.